## Строим BASELINE model - линейную регрессию

In [167]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np
from all_funcs import *
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error

In [168]:
def make_pipeline_preprocessor_and_model(num_col, cat_col, model_class, target=None, params=None, scaler=False, only_num_feat=False, only_cat_feat=False):
    num_col = num_col.to_list()
    num_col.remove('Listening_Time_minutes')
    if scaler:
        num_pipe = Pipeline([('imputer', SimpleImputer(strategy='mean')),
                             ('scaler', StandardScaler())])
    else:
        num_pipe = Pipeline([('imputer', SimpleImputer(strategy='mean'))])
    if only_num_feat:
        model = Pipeline([('preproc', num_pipe),
                      ('model', model_class(**params))])
        return model
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                         ('ohe', OneHotEncoder(drop='first',handle_unknown='ignore', sparse_output=False))])
    preprocessor = ColumnTransformer([('num', num_pipe, num_col),
                                      ('cat', cat_pipe, cat_col)])
    if params is not None:
        model = Pipeline([('preprocessor', preprocessor),
                      ('model', model_class(**params))])
    else:
        model = Pipeline([('preprocessor', preprocessor),
                      ('model', model_class())])
    if target is not None:
        imputer = SimpleImputer(strategy='mean')
        if isinstance(target, pd.Series):
            target = imputer.fit_transform(target.to_frame())
        else:
            target = imputer.fit_transform(target) #.values.reshape(-1,1)
        return model, imputer.fit_transform(target)
    return model


#валидируем предсказания и реальные таргетные значения
def show_all_regressor_metrics(y_pred, y_test):
    print('=============================================================================')
    print(f"R2 - {r2_score(y_test, y_pred)}")
    print(f"MSE - {mean_squared_error(y_test, y_pred)}")
    print(f"RMSE - {root_mean_squared_error(y_test, y_pred)}")
    print('=============================================================================')

In [169]:
#загружаем датасет
df = pd.read_csv('podcasts.csv')

In [181]:
df.isna().sum()

Podcast_Name                      0
Episode_Title                     0
Episode_Length_minutes         4695
Genre                             0
Host_Popularity_percentage        0
Publication_Day                   0
Publication_Time                  0
Guest_Popularity_percentage    4719
Number_of_Ads                     0
Episode_Sentiment                 0
Listening_Time_minutes            0
dtype: int64

In [182]:
df = df.dropna(subset=['Listening_Time_minutes', 'Guest_Popularity_percentage', 'Episode_Length_minutes'])

In [183]:
#собираем кат и кол-ые фичи
numeric_features_columns = df.select_dtypes(np.number).columns
cat_features_columns = df.select_dtypes(object).columns
target_column = 'Listening_Time_minutes'

In [184]:
# формируем тестовую и тренировочную выборки
X, y = df.drop(labels=['Listening_Time_minutes'], axis=1), df['Listening_Time_minutes']

#Создаем pipeline линейной регрессии с препроцессором фичей

model = make_pipeline_preprocessor_and_model(num_col=numeric_features_columns, cat_col=cat_features_columns, model_class=LinearRegression, scaler=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [185]:
#обучаем простую модель линейной регрессии
model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [186]:
#предсказываем таргет
y_pred = model.predict(X_test)

#### Метрики качества на TEST выборке

In [187]:
show_all_regressor_metrics(y_pred=y_pred, y_test=y_test)

R2 - 0.8282145774386761
MSE - 127.37472481346389
RMSE - 11.28604114884683


#### Метрики качества на TRAIN выборке

In [188]:
y_train_pred = model.predict(X_train)
show_all_regressor_metrics(y_pred=y_train, y_test=y_train_pred)


R2 - 0.7921579246269692
MSE - 127.0967256129819
RMSE - 11.273718357888043
